## Load the packages

In [26]:
from typing import TypedDict, Optional
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.types import interrupt
from langgraph.checkpoint.memory import InMemorySaver

from pydantic import BaseModel
from typing import Literal

## Load OpenAI

In [4]:
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
checkpointer = InMemorySaver()

## Define the agent and intent schema

In [21]:
class AgentState(TypedDict):
    user_input: str
    username: Optional[str]
    intent: Optional[str] # "info" | "request" | "complaint"
    response: Optional[str]
    reteieved_docs: Optional[list]

class IntentClassification(BaseModel):
    intent: Literal["info", "request", "complaint"]



In [76]:
async def classify_intent(state, llm, prompt):
    structured_llm = llm.with_structured_output(IntentClassification)
    result = await structured_llm.ainvoke(
        prompt.format(user_input=state["user_input"])
    )

    return result.intent

In [6]:
routing_eval_data = [
  {
    "user_input": "Can I renew my book?",
    "expected_output": "info"
  },
  {
    "user_input": "Please renew my book.",
    "expected_output": "request"
  },
  {
    "user_input": "Why can't I renew my book?",
    "expected_output": "complaint"
  },
  {
    "user_input": "Is it possible to reserve a book?",
    "expected_output": "info"
  },
  {
    "user_input": "Reserve this book for me.",
    "expected_output": "request"
  },
  {
    "user_input": "I tried to reserve this book but the system won't let me.",
    "expected_output": "complaint"
  },
  {
    "user_input": "How do I get a library card?",
    "expected_output": "info"
  },
  {
    "user_input": "I need a library card.",
    "expected_output": "request"
  },
  {
    "user_input": "My library card isn't being accepted.",
    "expected_output": "complaint"
  },
  {
    "user_input": "Can you tell me whether my membership is still active?",
    "expected_output": "info"
  },
  {
    "user_input": "Please renew my membership.",
    "expected_output": "request"
  },
  {
    "user_input": "My membership was renewed but my account still says expired.",
    "expected_output": "complaint"
  }
]

In [7]:
llm_mini = llm
llm_nano = ChatOpenAI(model="gpt-4.1-nano", temperature=0)

In [74]:
import asyncio


async def routing_evaluation(router, llm, prompt):
    async def evaluate_item(item, llm):
        state: AgentState = {
            "user_input": item["user_input"]
        }

        intent = await router(state, llm, prompt)

        return {
            "user_input": item["user_input"],
            "expected": item["expected_output"],
            "predicted": intent
        }

    results = await asyncio.gather(
        *(evaluate_item(item, llm) for item in routing_eval_data)
    )

    return results

### Test performance for prompts 

In [41]:
prompt1 = """
    Classify the user input info one of the three categories:
    1. info
    2. complaint
    3. request

    Input: {user_input}
    Respond with only one word: info, complaint, request
    info - User has some query related to the library
        Sample: 
            1 Who manages the library?
            2 Does the library have technology books?
            
    request - User is making some request 
        Sample:
            1 I want to reserve a book.
            2 I would like to register for a library membership.

    complaint - User wants to register some complaint
        Sample:
            1 The digital library is unavailable.
            2 My account shows the wrong due date.
    Note: These are queries related to a library.
    """

In [77]:
results = await routing_evaluation(classify_intent, llm_mini, prompt1)

In [78]:
y_true = [x["expected"] for x in results]
y_pred = [x["predicted"] for x in results]
y_true, y_pred

(['info',
  'request',
  'complaint',
  'info',
  'request',
  'complaint',
  'info',
  'request',
  'complaint',
  'info',
  'request',
  'complaint'],
 ['request',
  'request',
  'info',
  'request',
  'request',
  'complaint',
  'request',
  'request',
  'complaint',
  'request',
  'request',
  'complaint'])

### Confusion Matrix

In [79]:
from sklearn.metrics import confusion_matrix
import pandas as pd

labels = ["info", "request", "complaint"]

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=[f"Actual {label}" for label in labels],
    columns=[f"Predicted {label}" for label in labels]
)

print(cm_df)

                  Predicted info  Predicted request  Predicted complaint
Actual info                    0                  4                    0
Actual request                 0                  4                    0
Actual complaint               1                  0                    3


In [46]:
routing_evaluation(llm_nano, classify_intent, prompt1)

([('info', 'request'),
  ('request', 'request'),
  ('complaint', 'info'),
  ('info', 'request'),
  ('request', 'request'),
  ('complaint', 'complaint'),
  ('info', 'request'),
  ('request', 'request'),
  ('complaint', 'complaint'),
  ('info', 'request'),
  ('request', 'request'),
  ('complaint', 'complaint')],
 '7 / 12')

In [42]:
prompt2 = """
    You are an intent classifier for a public library AI assistant.
    
    Your task is to classify the user's message into exactly ONE of these intents:
    
    1. INFO
       The user wants information, an explanation, or an answer to a question.
       They are NOT asking the library to perform an action.
    
       Examples:
       - "What time does the library open?"
       - "Where is the library located?"
       - "Do you have books on Python?"
       - "Can I renew a book?"
       - "How do I get a library card?"
       - "Is book reservation available?"
    
    2. REQUEST
       The user wants the library or library system to perform an action
       on their behalf.
    
       Examples:
       - "Reserve this book for me."
       - "Please renew my book."
       - "I want to register for membership."
       - "Return this book for me."
       - "Please update my phone number."
       - "Cancel my book reservation."
    
    3. COMPLAINT
       The user is reporting a problem, failure, error, dissatisfaction,
       or something that went wrong with the library or its services.
    
       Examples:
       - "My library card is not working."
       - "I cannot log into my account."
       - "I tried to reserve a book but it failed."
       - "I returned my book but it still shows as borrowed."
       - "The library website is not working."
       - "The staff were rude to me."
    
    IMPORTANT RULES:
    
    - If the user is asking whether something is possible, available, allowed,
      or how something works, classify it as INFO.
    - If the user is explicitly asking the library to DO something, classify it
      as REQUEST.
    - If the user says that something is broken, failed, incorrect, unavailable,
      or did not work as expected, classify it as COMPLAINT.
    - Do not classify based only on keywords. Consider the user's actual intent.
    - A question about performing an action is INFO, not REQUEST.
    - An instruction to perform an action is REQUEST.
    - A failed attempt to perform an action is COMPLAINT.
    
    For example:
    
    "Can I renew my book?"
    → INFO
    
    "How can I renew my book?"
    → INFO
    
    "Please renew my book."
    → REQUEST
    
    "Renew my book."
    → REQUEST
    
    "I tried to renew my book but it failed."
    → COMPLAINT
    
    "Why can't I renew my book?"
    → COMPLAINT
    
    User message:
    {user_input}
    """

In [47]:
routing_evaluation(llm_mini, classify_intent, prompt2)

([('info', 'info'),
  ('request', 'request'),
  ('complaint', 'complaint'),
  ('info', 'info'),
  ('request', 'request'),
  ('complaint', 'complaint'),
  ('info', 'info'),
  ('request', 'request'),
  ('complaint', 'complaint'),
  ('info', 'info'),
  ('request', 'request'),
  ('complaint', 'complaint')],
 '12 / 12')

In [49]:
routing_evaluation(llm_nano, classify_intent, prompt2)

([('info', 'info'),
  ('request', 'request'),
  ('complaint', 'complaint'),
  ('info', 'info'),
  ('request', 'request'),
  ('complaint', 'complaint'),
  ('info', 'info'),
  ('request', 'request'),
  ('complaint', 'complaint'),
  ('info', 'info'),
  ('request', 'request'),
  ('complaint', 'complaint')],
 '12 / 12')